[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Status Codes


## What you will be able to do

Tell from a status code what happened to a request, including a code you have never seen, and
decide what a program should do about it: use the body, skip the resource, fix the request, stop,
or send the same request again later.


## The idea

### The problem

A program that calls an API a few thousand times will not get `200 OK` every time. A station has
been removed, a parameter is misspelled, a key has expired, the client has sent too many requests in
a minute, or the server is restarting. Each of those arrives as a response, and the part of a
response that means the same thing in every API is the three-digit status code at its start.

Code written only for `200` handles none of them well. The **Your First Request** notebook read a
`404` as if it were a station, and got `KeyError: 'name'` a line after the real problem. The error
named a symptom, and said nothing about whether to skip the station, fix the request or wait. Code
that saves whatever comes back does worse: it carries on, with an error message stored as data.

Those situations need different code. A station that no longer exists can be skipped. A misspelled
parameter has to be fixed, because the same request gets the same answer however often it is sent.
A server that is restarting may answer the same request a minute later.

### What a status code is

> A **status code** is the three-digit number in the status line of a response, saying how the
> server handled the request. Its first digit gives its **class**: `1xx` information, `2xx`
> success, `3xx` redirection, `4xx` client error, meaning the request was wrong, and `5xx` server
> error, meaning the server failed to handle it.

### Why it works that way

- **The status comes first.** It is on the first line of a response, ahead of the headers and the
  body, so a client can decide what to do before reading anything else. It is also sent before the
  body is finished, so a server that fails partway through a body has already said `200`. On 14
  September 2026, while this guide was being written, Open-Meteo's archive answered some requests
  with `200 OK` and a body that began "Unexpected error while streaming data".
- **The class says whose move it is.** A `4xx` means the request was wrong, so only a different
  request can succeed. `429 Too Many Requests` is the common exception: nothing in the request was
  wrong, there were too many of them, and the same request succeeds after a wait. A `5xx` means the
  server failed, so the same request may succeed later.
- **Code reads the number, and people read the phrase.** A server can send any phrase after the
  number, or none, and HTTP/2 carries no phrase at all. Phrases also change: RFC 9110, the current
  definition of HTTP, renamed `422 Unprocessable Entity` to `422 Unprocessable Content`, and Python's
  own table of phrases followed in version 3.13.
- **Every code has a class, even one no standard defines.** Cloudflare, which sits in front of many
  websites, sends codes from `520` to `527` when it cannot get a proper response from the website
  behind it. HTTP tells a client to treat a code it does not recognize as the `x00` code of its
  class, so a `520` is handled as a `500`.
- **An error's body may not come from the API.** An API's own errors usually explain themselves in
  JSON. A `502 Bad Gateway` or `504 Gateway Timeout` is usually written by a server in front of the
  API, when the API did not respond properly, and its body is often an HTML page.

### Where you will meet this

Every API's documentation lists the status codes its endpoints return, as the OpenAPI document in
the **Exploring an API** notebook did for a station: `200` and `404`. The Network tab of a browser's
developer tools and Postman's response panel both show the status code before anything else.
GitHub's API responds `403` or `429` to a client over its rate limit, and FastAPI, which the **Your
First API Server** notebook uses, responds `422` to a request that fails validation.

### What this notebook covers

- Reading the status code before the body
- The five classes, and `HTTPStatus`, which names the numbers
- A success with no body
- `4xx` codes: a request that has to change, two kinds of `404`, `401` and `403`, and `429` with
  its `Retry-After` header
- `5xx` codes, including a gateway's HTML where JSON was expected
- A code no standard defines, decided by its class
- `raise_for_status`, with the response still in reach
- A decision for every response, in one function
- Four errors, from parsing a gateway's page as JSON to a code `HTTPStatus` does not know

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

CLASSES = {2: "the request worked", 4: "the request was wrong", 5: "the server failed"}

for path in ["/stations/tromso", "/stations/narvik", "/status/503"]:
    response = requests.get(f"http://127.0.0.1:8765{path}", timeout=10)
    print(response.status_code, response.reason, "->", CLASSES[response.status_code // 100])
```

```
200 OK -> the request worked
404 Not Found -> the request was wrong
503 Service Unavailable -> the server failed
```

Three responses, and the first digit of each status code says which of three things happened.
`/status/503` is an address on the practice API that fails on purpose, and the worked examples use
addresses like it to meet the codes a real server sends only when something goes wrong.


## Setup

Seven imports, the last of them the practice API.

- `HTTPStatus` names every registered status code, with its phrase and its class
- `requests` sends every request in this notebook
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import importlib
import sys
import urllib.request
from http import HTTPStatus
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### Reading the status code before the body

The practice API has an endpoint for this notebook. `/status/<code>` responds with the status code
its path names, and with the headers and body a real server sends with that code, so a program can
meet a `503` without waiting for a real server to fail. Public testing services such as httpbin
offer the same kind of endpoint.

Here are four addresses: a station, a station that does not exist, and two that stand in for a
server having a bad day. The program reads the status code of each response first, and the status
code decides what happens to the body:


In [2]:
addresses = [f"{BASE}/stations/tromso", f"{BASE}/stations/narvik",
             f"{BASE}/status/503", f"{BASE}/status/502"]

for address in addresses:
    response = requests.get(address, timeout=10)
    code = response.status_code
    if code == HTTPStatus.OK:
        print(response.json()["name"])
    elif code == HTTPStatus.NOT_FOUND:
        print(f"{code}: skip it, {response.json()['error']}")
    elif code == HTTPStatus.SERVICE_UNAVAILABLE:
        print(f"{code}: try again in {response.headers['Retry-After']} seconds")
    else:
        print(f"{code} {response.reason}: the server failed, so try again later")


Tromso
404: skip it, no station with id 'narvik'
503: try again in 120 seconds
502 Bad Gateway: the server failed, so try again later


Each response got the handling its status code called for. Only the `200` was read as a station.
The `404`'s body was read for its error message, the `503`'s `Retry-After` header said how long to
wait, and the `502`'s body was never parsed, which matters, because it is an HTML page rather than
JSON. The sections that follow take those decisions apart, class by class.

### Five classes, told apart by the first digit

requests puts the number from the status line in `status_code`, and its phrase in `reason`. The
class is the first digit, which integer division by 100 gives:


In [3]:
for code in [200, 201, 204, 301, 400, 404, 429, 500, 503]:
    response = requests.get(f"{BASE}/status/{code}", allow_redirects=False, timeout=10)
    print(f"{response.status_code} {response.reason:<21} class {response.status_code // 100}xx  ok={response.ok}")


200 OK                    class 2xx  ok=True
201 Created               class 2xx  ok=True
204 No Content            class 2xx  ok=True
301 Moved Permanently     class 3xx  ok=True
400 Bad Request           class 4xx  ok=False
404 Not Found             class 4xx  ok=False
429 Too Many Requests     class 4xx  ok=False
500 Internal Server Error class 5xx  ok=False
503 Service Unavailable   class 5xx  ok=False


`ok` is `True` for every code below 400, so it says only that a response is not an error: the `301`
is `ok`, and carries an address rather than a body. `allow_redirects=False` kept requests from
following it, as the **Your First Request** notebook showed. No `1xx` code appears: those are
provisional, sent ahead of the final response, and client code almost never sees one.

### Names for the numbers: HTTPStatus

A `429` in code is a number the next reader has to recognize. The standard library's
`http.HTTPStatus` gives every registered code a name:


In [4]:
status = HTTPStatus(429)

print(repr(status))
print(status.value, status.phrase)
print(status.description)
print("client error:", status.is_client_error, "| server error:", status.is_server_error)
print("equal to 429:", status == 429)


<HTTPStatus.TOO_MANY_REQUESTS: 429>
429 Too Many Requests
The user has sent too many requests in a given amount of time ("rate limiting")
client error: True | server error: False
equal to 429: True


Each member is also an integer, equal to its number, so
`response.status_code == HTTPStatus.TOO_MANY_REQUESTS` works, and says what it tests.
`is_client_error` and `is_server_error`, along with `is_success`, `is_redirection` and
`is_informational`, test the class; they arrived in Python 3.12. Here are the names of the codes
this notebook meets:


In [5]:
for code in [200, 204, 400, 401, 403, 404, 429, 500, 502, 503, 504]:
    status = HTTPStatus(code)
    kind = "success" if status.is_success else "client error" if status.is_client_error else "server error"
    print(f"{code}  HTTPStatus.{status.name:<22} {kind}")


200  HTTPStatus.OK                     success
204  HTTPStatus.NO_CONTENT             success
400  HTTPStatus.BAD_REQUEST            client error
401  HTTPStatus.UNAUTHORIZED           client error
403  HTTPStatus.FORBIDDEN              client error
404  HTTPStatus.NOT_FOUND              client error
429  HTTPStatus.TOO_MANY_REQUESTS      client error
500  HTTPStatus.INTERNAL_SERVER_ERROR  server error
502  HTTPStatus.BAD_GATEWAY            server error
503  HTTPStatus.SERVICE_UNAVAILABLE    server error
504  HTTPStatus.GATEWAY_TIMEOUT        server error


requests has names of its own, such as `requests.codes.too_many_requests`, but `HTTPStatus` is part
of Python and works with any HTTP library, `urllib` included.

### A success with no body: 204 No Content

Not every success has a body. `204 No Content` means the request worked and there is nothing to
send back, which is how many APIs respond to a `DELETE`:


In [6]:
done = requests.get(f"{BASE}/status/204", timeout=10)

print(done.status_code, done.reason, "| ok:", done.ok)
print("body:", done.content)
print("Content-Type:", done.headers.get("Content-Type"))
print("equal to 200:", done.status_code == HTTPStatus.OK)


204 No Content | ok: True
body: b''
Content-Type: None
equal to 200: False


The body is empty, so `json()` has nothing to parse and raises, as it does in Common errors for a
body that is not JSON. And a success is not always `200`: code that checks `status_code == 200`
treats this `204` as a failure, and does the same to the `201 Created` that the **Sending Data**
notebook meets. Check the class, `status_code // 100 == 2`, and read a body only when there is one.

### A 4xx: the request has to change

A `4xx` means the server understood the request well enough to say it was wrong, so the same
request gets the same answer. Here is a real one: Open-Meteo's archive, asked twice for a daily
variable it does not have.


In [7]:
query = {"latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15",
         "end_date": "2025-01-17", "daily": "mean_temperature", "models": "era5"}

for attempt in [1, 2]:
    response = requests.get(OPEN_METEO, params=query, timeout=30)
    print(attempt, response.status_code, response.reason, "|", response.json()["reason"])


1 400 Bad Request | Invalid value: Cannot initialize ForecastVariableDaily from invalid String value mean_temperature


2 400 Bad Request | Invalid value: Cannot initialize ForecastVariableDaily from invalid String value mean_temperature


The same request, the same `400`, and a reason that names the value at fault: Open-Meteo calls the
variable `temperature_2m_mean`. A third attempt would get a third `400`, and so would waiting. The
mistake is in the request, so a program should report the reason and stop, or change the request.

Other `4xx` codes work the same way. `405 Method Not Allowed`, which the **What an API Is** notebook
met, needs a different method, and its `Allow` header lists the ones that work. Some APIs, FastAPI
among them, send `422 Unprocessable Content` for a request that is well formed but holds an invalid
value; to a client it means the same as this `400`.

### Two kinds of 404

`404 Not Found` covers two situations that need different code. Here is one of each:


In [8]:
for path in ["/stations/narvik", "/station/tromso"]:
    response = requests.get(f"{BASE}{path}", timeout=10)
    print(response.status_code, path, "|", response.json()["error"])


404 /stations/narvik | no station with id 'narvik'
404 /station/tromso | nothing at /station/tromso


The same status code, and two different problems. The first is an answer: the practice API has
stations, Narvik is not one of them, and a program asking about Narvik can skip it. The second is a
bug: `station` is missing its `s`, so the address matches no endpoint, and every station requested
that way would come back `404` too. A program that skips every `404` would report that the API had
no stations at all. This API's bodies tell the two apart; when an API's bodies do not, compare the
address with its documentation.

### 401 and 403: who is asking, and what they may do

Two `4xx` codes are about credentials. The **Authentication** notebook sends the keys that avoid
them:


In [9]:
for code in [401, 403]:
    response = requests.get(f"{BASE}/status/{code}", timeout=10)
    print(response.status_code, response.reason, "|", response.json()["error"])
    print("    WWW-Authenticate:", response.headers.get("WWW-Authenticate"))


401 Unauthorized | the request needs a valid API key
    WWW-Authenticate: Bearer
403 Forbidden | the API key does not allow this request
    WWW-Authenticate: None


`401 Unauthorized` means the server does not know who is asking: the credentials are missing or
wrong, and the `WWW-Authenticate` header names the kind it expects. `403 Forbidden` means the server
knows who is asking, and will not allow the request. The phrase for `401` is misleading, because
that code is about who you are, and `403` is the one about what you may do. Neither changes on its
own, so a program should stop and say which one it met: the fix is a key or a permission, and no
number of retries provides either.

### 429 Too Many Requests, and Retry-After

`429` is the `4xx` where nothing in the request was wrong: the client sent too many requests, too
quickly. The server usually says how long to wait, in a `Retry-After` header:


In [10]:
limited = requests.get(f"{BASE}/status/429", timeout=10)

print(limited.status_code, limited.reason, "|", limited.json()["error"])
retry_after = limited.headers["Retry-After"]
print("Retry-After:", repr(retry_after), "->", int(retry_after), "seconds")


429 Too Many Requests | too many requests: wait 30 seconds before sending another
Retry-After: '30' -> 30 seconds


The same request can succeed after the wait, so a `429` calls for waiting rather than changing
anything. `Retry-After` is a string, like every header value, and needs `int` before it can be used
as a number of seconds. It can also be a date, and many APIs send headers of their own counting the
requests left. The **Rate Limits** notebook handles both, and paces requests so that a `429` does not
happen.

### A 5xx: the server failed

A `5xx` means the request may well have been fine, and the server could not handle it. Here are
four, with what arrives with each:


In [11]:
for code in [500, 502, 503, 504]:
    response = requests.get(f"{BASE}/status/{code}", timeout=10)
    print(response.status_code, response.reason)
    print("    Content-Type:", response.headers["Content-Type"],
          "| Retry-After:", response.headers.get("Retry-After"))


500 Internal Server Error
    Content-Type: application/json | Retry-After: None
502 Bad Gateway
    Content-Type: text/html; charset=utf-8 | Retry-After: None
503 Service Unavailable
    Content-Type: application/json | Retry-After: 120
504 Gateway Timeout
    Content-Type: text/html; charset=utf-8 | Retry-After: None


- `500 Internal Server Error` is the API's own failure, often a bug. Here the API wrote the
  response itself, in the JSON its other errors use.
- `502 Bad Gateway` and `504 Gateway Timeout` come from a gateway, a server in front of the API such
  as a proxy or a load balancer, when the API behind it sent a broken response or none in time. The
  API did not write them, so their bodies are the gateway's HTML.
- `503 Service Unavailable` means the server cannot handle requests for now, during maintenance or
  under too much load, and `Retry-After` says for how long.

Here is the page a gateway sent where JSON was expected:


In [12]:
gateway = requests.get(f"{BASE}/status/502", timeout=10)

print(gateway.text)


<!doctype html>
<html>
<head><title>502 Bad Gateway</title></head>
<body><h1>502 Bad Gateway</h1><p>The server in front of the API received an invalid response from it.</p></body>
</html>



`json()` on this page raises `JSONDecodeError`, which Common errors shows, so check the
`Content-Type` before parsing. Every `5xx` can clear up without the client changing anything, so the
same request may succeed later. Which failures are worth retrying, how often and how far apart, is
the subject of the **Errors and Retries** notebook.

### A code no standard defines

A server can send any three digits, and `HTTPStatus` knows only the registered ones. Here is `520`,
the code Cloudflare sends when the website behind it returns a response it cannot use:


In [13]:
unknown = requests.get(f"{BASE}/status/520", timeout=10)

print(unknown.status_code, repr(unknown.reason), "| ok:", unknown.ok)
print("registered:", unknown.status_code in HTTPStatus)
print("class:", f"{unknown.status_code // 100}xx")


520 '' | ok: False
registered: False
class: 5xx


No phrase came with it, and `HTTPStatus` has no name for it, but it still has a class. HTTP's rule
for a client meeting a code it does not recognize is to treat it as the `x00` code of its class, so
this `520` gets the handling of a `500`. Looking it up with `HTTPStatus(520)` raises instead, as
Common errors shows, while code that falls back on the first digit handles any code a server sends.

### raise_for_status, with the response still in reach

The **Your First Request** notebook used `raise_for_status()` to stop at an error status. The
exception it raises carries the response, so a handler can still read the status code, the headers
and the body:


In [14]:
for path in ["/stations/oslo", "/stations/narvik", "/status/503"]:
    try:
        response = requests.get(f"{BASE}{path}", timeout=10)
        response.raise_for_status()
    except requests.HTTPError as error:
        print(error)
        print("    status:", error.response.status_code,
              "| Retry-After:", error.response.headers.get("Retry-After"))
    else:
        print(response.json()["name"])


Oslo
404 Client Error: Not Found for url: http://127.0.0.1:8765/stations/narvik
    status: 404 | Retry-After: None
503 Server Error: Service Unavailable for url: http://127.0.0.1:8765/status/503
    status: 503 | Retry-After: 120


`error.response` is the Response that `get` returned. requests also names the class in its message:
a `4xx` is a Client Error and a `5xx` a Server Error. This shape suits code where most calls should
simply work: each request is followed by a single line, and every decision about failures is made in
one place, the handler.

### A decision for every response

Everything in this notebook, in one function. `decide` takes any response and returns what a program
should do about it, with the detail needed to do it. `error_message` reads what an error response
says went wrong, without assuming its body is JSON:


In [15]:
def error_message(response):
    """What an error response says went wrong, read from its JSON body when it has one."""
    content_type = response.headers.get("Content-Type", "")
    if not content_type.startswith("application/json"):
        return f"{content_type} instead of JSON"
    body = response.json()
    return body.get("reason") or body.get("error")    # Open-Meteo says reason, the practice API error


def decide(response):
    """What to do about a response: an action, and the detail needed to take it."""
    code = response.status_code
    if code // 100 == 2:
        return "use", ("the body" if response.content else "no body")
    if code == HTTPStatus.NOT_FOUND:
        return "skip", error_message(response)
    if code in (HTTPStatus.UNAUTHORIZED, HTTPStatus.FORBIDDEN):
        return "stop", error_message(response)
    if code in (HTTPStatus.TOO_MANY_REQUESTS, HTTPStatus.SERVICE_UNAVAILABLE):
        return "wait", f"{int(response.headers.get('Retry-After', 60))} seconds"
    if code // 100 == 4:
        return "fix the request", error_message(response)
    if code // 100 == 5:
        return "try later", error_message(response)
    return "stop", f"no rule for status {code}"


Here it is against one response of every kind this notebook met:


In [16]:
cases = [
    ("a station",                   f"{BASE}/stations/tromso", None),
    ("a success with no body",      f"{BASE}/status/204", None),
    ("a station that is not there", f"{BASE}/stations/narvik", None),
    ("a mistyped address",          f"{BASE}/station/tromso", None),
    ("a variable Open-Meteo lacks", OPEN_METEO, query),
    ("no API key",                  f"{BASE}/status/401", None),
    ("too many requests",           f"{BASE}/status/429", None),
    ("a server bug",                f"{BASE}/status/500", None),
    ("a gateway failure",           f"{BASE}/status/502", None),
    ("maintenance",                 f"{BASE}/status/503", None),
    ("a code no standard defines",  f"{BASE}/status/520", None),
]

for label, address, params in cases:
    response = requests.get(address, params=params, timeout=30)
    action, detail = decide(response)
    print(f"{response.status_code}  {label:<28} {action:<16} {detail}")


200  a station                    use              the body
204  a success with no body       use              no body
404  a station that is not there  skip             no station with id 'narvik'
404  a mistyped address           skip             nothing at /station/tromso


400  a variable Open-Meteo lacks  fix the request  Invalid value: Cannot initialize ForecastVariableDaily from invalid String value mean_temperature
401  no API key                   stop             the request needs a valid API key
429  too many requests            wait             30 seconds
500  a server bug                 try later        the server failed while handling the request
502  a gateway failure            try later        text/html; charset=utf-8 instead of JSON
503  maintenance                  wait             120 seconds
520  a code no standard defines   try later        status 520


### Where each part came from

| In the program | What it relies on | The section that showed it |
|---|---|---|
| `code // 100 == 2` | the class is the first digit | Five classes, told apart by the first digit |
| `HTTPStatus.NOT_FOUND` | a name for the number | Names for the numbers: HTTPStatus |
| `"the body" if response.content else "no body"` | a success can have no body | A success with no body: 204 No Content |
| `"skip"` for a `404` | a `404` can be an answer | Two kinds of 404 |
| `"stop"` for `401` and `403` | credentials do not change on their own | 401 and 403: who is asking, and what they may do |
| `int(response.headers.get("Retry-After", 60))` | a wait in seconds, sent as a string | 429 Too Many Requests, and Retry-After |
| `"fix the request"` for any other `4xx` | the same request gets the same answer | A 4xx: the request has to change |
| `"try later"` for any other `5xx`, `520` included | a failed server may recover, and an unknown code has a class | A 5xx: the server failed, and A code no standard defines |
| `error_message` checking `Content-Type` first | a gateway's HTML where JSON was expected | A 5xx: the server failed |

One row is wrong, on purpose. The mistyped address was skipped as though it were a missing station,
because `decide` reads only the status code, which is all that every API has in common. The detail
column shows the difference, `nothing at /station/tromso`. A client written for one API can read
that API's bodies and tell the two `404`s apart, and the **A Real Client** notebook builds a client
of that kind.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/04-status-codes-solutions.ipynb).

**1.** Request `/status/451` from the practice API, and print its status code, its reason, and its
class as a single digit.


In [17]:
# your code here


**2.** Use `HTTPStatus` to print the name and the phrase of `409` and of `410`, and whether each is a
client error.


In [18]:
# your code here


**3.** Request `/status/503` and `/status/429`, and print how many seconds the two responses ask a
client to wait, added together.


In [19]:
# your code here


**4.** Write `json_or_none(response)`, which returns the parsed body when the status code is `2xx`
and the `Content-Type` starts with `application/json`, and returns `None` otherwise. Try it on
`/stations/oslo`, `/status/204`, `/stations/narvik` and `/status/504`.


In [20]:
# your code here


**5.** Write `class_name(code)`, which returns `"informational"`, `"success"`, `"redirection"`,
`"client error"` or `"server error"` from the first digit of any status code, registered or not.
Print it for `204`, `308`, `451`, `520` and `599`.


In [21]:
# your code here


**6.** `410 Gone` means a resource has been removed and will not return. Write
`decide_gone(response)`, which returns `("stop asking", error_message(response))` for a `410`, and
whatever `decide(response)` returns for anything else. Try it on `/status/410` and `/status/404`.


In [22]:
# your code here


## Common errors

### JSONDecodeError: Expecting value: line 1 column 1 (char 0)


In [23]:
gateway = requests.get(f"{BASE}/status/502", timeout=10)
gateway.json()


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

The body is the gateway's HTML page. Its first character, `<`, cannot begin a JSON value, and "line 1
column 1 (char 0)" is where the parser stopped. An empty body, like a `204`'s, raises the same error.
The message is about JSON because `json()` is where the problem surfaced, but the cause is the
status code. Check it, and the `Content-Type`, before parsing:


In [24]:
if gateway.ok and gateway.headers.get("Content-Type", "").startswith("application/json"):
    print(gateway.json())
else:
    print("not parsed:", gateway.status_code, gateway.reason, "|", gateway.headers.get("Content-Type"))


not parsed: 502 Bad Gateway | text/html; charset=utf-8


### KeyError: 'retry-after'


In [25]:
failed = requests.get(f"{BASE}/status/500", timeout=10)
wait = int(failed.headers["Retry-After"])


KeyError: 'retry-after'

Not every failure says how long to wait, and this `500` has no `Retry-After` header. A missing
header raises `KeyError`, as a missing dictionary key does, and requests looks header names up in
lowercase, which is why the message says `'retry-after'`. Use `get`, with a default the program
chooses:


In [26]:
wait = int(failed.headers.get("Retry-After", 60))
print(failed.status_code, "| wait", wait, "seconds")


500 | wait 60 seconds


### TypeError: can only concatenate str (not "int") to str


In [27]:
limited = requests.get(f"{BASE}/status/429", timeout=10)
wait = limited.headers["Retry-After"] + 5      # a few seconds more than asked, to be safe


TypeError: can only concatenate str (not "int") to str

Header values are always strings, so this adds the number `5` to the string `'30'`, which raises
`TypeError`. Convert the header first:


In [28]:
wait = int(limited.headers["Retry-After"]) + 5
print(wait)


35


### ValueError: 520 is not a valid HTTPStatus


In [29]:
unknown = requests.get(f"{BASE}/status/520", timeout=10)
print(HTTPStatus(unknown.status_code).phrase)


ValueError: 520 is not a valid HTTPStatus

`HTTPStatus` holds only registered codes, and a server can send any three digits, so code that turns
every status code into a member raises at the first unregistered one. Look a code up only when it is
registered, and describe it by its class when it is not:


In [30]:
code = unknown.status_code
print(code, HTTPStatus(code).phrase if code in HTTPStatus else f"an unregistered {code // 100}xx")


520 an unregistered 5xx


## Recap

- A status code's first digit is its class: `2xx` the request worked, `4xx` it was wrong, and `5xx`
  the server failed.
- A `4xx` needs a different request, except `429`, which needs the same request after a wait. A
  `5xx` may clear up on its own.
- Compare numbers, not phrases. `HTTPStatus` names the registered codes, and the first digit decides
  any code, registered or not.
- Check the status code and the `Content-Type` before reading a body: a `204` has none, and a `502`
  or `504` usually carries a gateway's HTML.
- `Retry-After`, sent with many `429` and `503` responses, is a number of seconds in a string, and is
  missing from most other responses.
- The same `404` can be an answer or a bug in the address, and only the body or the documentation
  says which.


## What is next

The **Query Parameters** notebook. A query value Open-Meteo did not recognize brought a `400` here;
that notebook builds queries with `params` in full, including the values that need encoding.


---

&#8592; **Previous:** [Your First Request](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/03-your-first-request.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Query Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/05-query-parameters.ipynb) &#8594;
